# Đo độ trễ inference thật: prefill, decode, TTFT

Mục 4 và 5 lý thuyết nói inference có hai pha: **prefill** (đọc cả prompt một lần, nhanh) và **decode** (sinh từng
token, chậm), và độ trễ người dùng thấy là $$T \approx \text{TTFT} + n_{out} \cdot \text{TPOT}$$. Bài này đo điều đó trên
`qwen2.5:3b` chạy bằng Ollama, với prompt là điều luật thật của Zalo Legal, rồi tự dựng một mô hình độ trễ tuyến tính
để dự đoán độ trễ của một tính năng trước khi xây nó.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` gọi Ollama thật; test bỏ qua chúng. Cần `ollama serve` và `ollama pull qwen2.5:3b`; nếu Ollama
  chạy ở máy khác, đặt `OLLAMA_HOST` (như `llmkit`).
- Độ trễ đo được phụ thuộc máy và việc GPU có đang bận việc khác không: **số của bạn sẽ khác số trong lời giải**, nhưng
  các quan hệ (prefill nhanh hơn decode nhiều lần, độ trễ tăng tuyến tính theo số token output) phải giống.
- Chấm: `uv run pytest tracks/ai-engineer/modules/00-introduction/exercises/intermediate/02-inference-latency -q`

In [ ]:
import math

import numpy as np

## 1. Đọc các trường thời gian Ollama trả về

Response không stream của `POST /api/chat` có các trường (thời gian tính bằng **nano giây**):

| Trường | Nghĩa |
| --- | --- |
| `load_duration` | nạp model vào bộ nhớ (lớn ở lần gọi đầu: cold start; gần 0 khi model đã nạp) |
| `prompt_eval_count`, `prompt_eval_duration` | số token prompt và thời gian **prefill** |
| `eval_count`, `eval_duration` | số token sinh ra và thời gian **decode** |
| `total_duration` | tổng thời gian phía server |

`ollama_metrics(resp) -> dict` trả các khoá: `n_in`, `n_out` (int), `load_s`, `prefill_s`, `decode_s`, `total_s`
(giây, float), `prefill_tps = n_in / prefill_s`, `decode_tps = n_out / decode_s` (token/giây; thời gian bằng 0 thì
`0.0`), và `ttft_s = load_s + prefill_s` (ước lượng TTFT phía server, chưa tính mạng). Trường nào thiếu thì coi là 0
(Ollama bỏ `load_duration` khi không phải nạp gì).

In [ ]:
NS = 1e9


def ollama_metrics(resp: dict) -> dict:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: kết nối Ollama, nạp dữ liệu Zalo Legal làm prompt thật
import json
import os
import sys
import time
from pathlib import Path

import httpx
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
if not HOST.startswith("http"):
    HOST = "http://" + HOST
MODEL = "qwen2.5:3b"
client = httpx.Client(base_url=HOST, timeout=600)
print("Ollama", client.get("/api/version").json())

ZALO = ensure("zalo-legal")
with open(ZALO / "corpus.jsonl", encoding="utf-8") as f:
    LONG = [d for d in map(json.loads, f) if len(d["text"]) > 9000][:40]  # 40 điều dài nhất đầu tiên, đủ cắt mọi cỡ
print(len(LONG), "điều luật dài làm prompt")


def call(text_chars: int, n_out: int, i: int) -> dict:
    # Mỗi request một điều luật KHÁC nhau: Ollama giữ KV cache của prompt trước, prompt trùng đầu sẽ prefill nhanh giả.
    doc = LONG[i % len(LONG)]
    messages = [{"role": "user", "content": "Tóm tắt điều luật sau:\n" + doc["text"][:text_chars]}]
    body = {"model": MODEL, "stream": False, "messages": messages,
            "options": {"temperature": 0, "seed": 0, "num_predict": n_out}}
    resp = client.post("/api/chat", json=body).json()
    resp.pop("message", None)
    return resp


first = ollama_metrics(call(200, 8, 39))
print("lần gọi đầu (có thể là cold start):", {k: round(v, 3) for k, v in first.items()})

In [ ]:
# explore: quét độ dài prompt (output cố định 64 token) và độ dài output (prompt khoảng 300 token), 2 lần mỗi cỡ
rows, i = [], 0
for chars in [200, 1000, 3000, 6000, 9000]:
    for _ in range(2):
        rows.append({"sweep": "input", **ollama_metrics(call(chars, 64, i))})
        i += 1
for n_out in [16, 64, 256]:
    for _ in range(2):
        rows.append({"sweep": "output", **ollama_metrics(call(1000, n_out, i))})
        i += 1
df = pd.DataFrame(rows)
# Thời gian GPU thật sự làm việc cho request này, và phần còn lại của total_duration: chờ trong hàng đợi của Ollama
# (máy dùng chung, request khác đang chạy). Trên máy riêng, queue_s gần 0.
df["service_s"] = df.load_s + df.prefill_s + df.decode_s
df["queue_s"] = df.total_s - df.service_s
cols = ["n_in", "n_out", "prefill_s", "decode_s", "service_s", "queue_s", "total_s", "prefill_tps", "decode_tps"]
display(df.groupby(["sweep", "n_out"])[cols].median().round(3))
print("prefill nhanh hơn decode (trung vị):", round(df.prefill_tps.median() / df.decode_tps.median(), 1), "lần")

## 2. Phân vị và mô hình độ trễ tuyến tính

- `summarize(values) -> dict`: `n`, `mean`, `p50`, `p95`, `max` của một dãy số (phân vị nội suy tuyến tính, như
  `np.percentile` mặc định). Dãy rỗng thì `ValueError`.
- `fit_latency(n_in, n_out, total_s) -> dict`: khớp bình phương tối thiểu $$T \approx a + b \cdot n_{in} + c \cdot n_{out}$$
  trên các phép đo, trả `{"a", "b", "c", "r2"}` (float), với $$R^2 = 1 - \frac{\sum (T - \hat T)^2}{\sum (T - \bar T)^2}$$.
  Ba dãy phải cùng độ dài, ít nhất 3 phép đo, ngược lại `ValueError`.
- `predict_latency(model, n_in, n_out) -> float`: $$a + b\,n_{in} + c\,n_{out}$$.

$$b$$ ước lượng giây cho mỗi token prompt (nghịch đảo tốc độ prefill), $$c$$ giây cho mỗi token output (TPOT).

In [ ]:
def summarize(values) -> dict:
    # TODO
    raise NotImplementedError


def fit_latency(n_in, n_out, total_s) -> dict:
    # TODO
    raise NotImplementedError


def predict_latency(model: dict, n_in: float, n_out: float) -> float:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: khớp mô hình trên thời gian phục vụ (bỏ thời gian xếp hàng), so dự đoán với đo thật
import matplotlib.pyplot as plt

model = fit_latency(df.n_in, df.n_out, df.service_s)
print("khớp trên total_s (gồm cả xếp hàng) để so:", {k: round(v, 5) for k, v in fit_latency(df.n_in, df.n_out, df.total_s).items()})
print({k: round(v, 5) for k, v in model.items()})
print(f"=> prefill ≈ {1 / model['b']:.0f} token/giây, decode ≈ {1 / model['c']:.1f} token/giây theo mô hình")
pred = [predict_latency(model, a, b) for a, b in zip(df.n_in, df.n_out)]
fig, ax = plt.subplots(figsize=(4.5, 4))
for sweep, g in df.assign(pred=pred).groupby("sweep"):
    ax.scatter(g.pred, g.service_s, label=f"quét {sweep}")
lim = [0, max(df.service_s.max(), max(pred)) * 1.05]
ax.plot(lim, lim, "k--", lw=1)
ax.set_xlabel("dự đoán (giây)")
ax.set_ylabel("đo thật load + prefill + decode (giây)")
ax.legend()
plt.show()

## 3. Đo phía client khi stream

Người dùng không thấy `prompt_eval_duration`; họ thấy chữ đầu tiên hiện lên sau bao lâu. Khi stream, client ghi thời
điểm gửi và thời điểm nhận **từng chunk có nội dung** (mỗi chunk của Ollama thường là một token).

`stream_timing(t_send, chunk_times) -> dict` (thời điểm tính bằng giây, cùng một đồng hồ):
`ttft_s` (chunk đầu trừ `t_send`), `e2e_s` (chunk cuối trừ `t_send`), `tpot_s` (thời gian trung bình giữa hai chunk
liên tiếp: (chunk cuối − chunk đầu) / (số chunk − 1); một chunk thì `0.0`). Không có chunk nào thì `ValueError`.

In [ ]:
def stream_timing(t_send: float, chunk_times: list[float]) -> dict:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: 8 request stream, đo TTFT và TPOT phía client, báo p50/p95
def stream_once(i: int, n_out: int = 64) -> dict:
    doc = LONG[(i + 7) % len(LONG)]
    body = {"model": MODEL, "stream": True,
            "messages": [{"role": "user", "content": "Tóm tắt điều luật sau:\n" + doc["text"][:2000]}],
            "options": {"temperature": 0, "seed": 0, "num_predict": n_out}}
    times, last = [], None
    t_send = time.perf_counter()
    with client.stream("POST", "/api/chat", json=body) as r:
        for line in r.iter_lines():
            if not line:
                continue
            chunk = json.loads(line)
            if chunk.get("message", {}).get("content"):
                times.append(time.perf_counter())
            if chunk.get("done"):
                last = chunk
    return {**stream_timing(t_send, times), "server_ttft_s": ollama_metrics(last)["ttft_s"]}


streams = pd.DataFrame([stream_once(i) for i in range(8)])
for col in ["ttft_s", "tpot_s", "e2e_s", "server_ttft_s"]:
    print(col, {k: round(v, 3) for k, v in summarize(streams[col]).items()})

In [ ]:
# explore: dự đoán trước khi xây: RAG trả lời pháp luật, prompt 2 000 token, câu trả lời 300 token
for n_in, n_out in [(2000, 300), (2000, 80), (500, 300)]:
    print(f"n_in={n_in:5d} n_out={n_out:4d}: dự đoán {predict_latency(model, n_in, n_out):.1f} giây")

## Đọc kết quả

- Tốc độ prefill gấp bao nhiêu lần decode trên máy bạn? Khớp với giải thích "compute-bound và memory-bound" ở mục 4 lý
  thuyết không?
- Trong mô hình $$a + b\,n_{in} + c\,n_{out}$$, muốn giảm độ trễ của câu trả lời RAG (2 000 token vào, 300 token ra) thì
  cắt bớt tài liệu hay rút ngắn câu trả lời có lợi hơn? Gấp bao nhiêu lần?
- `queue_s` (total trừ thời gian phục vụ) lớn cỡ nào trên máy bạn? Khớp mô hình trên `total_s` thay vì `service_s` thì hệ
  số và $$R^2$$ đổi ra sao, và vì sao?
- TTFT đo phía client lệch TTFT phía server bao nhiêu? Phần lệch gồm những gì?
- Chạy lại cell quét khi một chương trình khác đang dùng GPU (hoặc chạy hai notebook cùng lúc): p95 thay đổi thế nào
  so với p50?

Sau đó làm phần trắc nghiệm của bài trong app.